# rail3D design review

Module-by-module visualizations of the 3D upgrade for approval **before** full data generation and training.
Physics gates V1–V4 (solver equivalence vs the verbatim Face3D code, FFT vs conv2d propagation,
angular-spectrum cross-check, sanity checks) are automated in `tests_physics_3d.py` and were all PASS —
see `data/generated/verification_report.json`.

Run top-to-bottom (CPU-safe on the laptop; set `DEVICE='cuda:0'` for the field maps to go faster).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
import torch
from rail3d import config, viz_setup

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
config.ensure_dirs()
print('device:', DEVICE)

## 1. Physical setup
λ = 5 mm (60 GHz), 60×30 grid at dx = 2.5 mm (150×75 mm aperture), horn at 140 mm / 55° in the x–z plane,
crown → metasurface 150 mm (= 30λ), metasurface → detectors 100 mm (= 20λ). This is the experimentally
validated Face3D λ=8 scene scaled by 5/8 wherever Face3D chose lengths in wavelengths, so every Fresnel
number is preserved; the rail and its defects keep their physical mm sizes. Check every dimension and
orientation — all of them are derived in `config.py`, so this cell must agree with the printout below.

In [ ]:
viz_setup.setup_diagram(save_path=config.FIGURE_DIR / 'setup_diagram.png');

## 2. Geometry: cross-sections and swept meshes
Defect CSV loops are width-matched to the intact reference (same convention as the 2D pipeline). Since
rev.2 the defect is a **per-point depth field** `displaced(s,y) = intact(s) − d(s,y)·n̂(s)`, not a
per-slice blend of whole cross-sections — that older model could only make defects uniform across the
head. Per-class parametric generators (`sample_defect_params` → `render_depth_field`) draw
resolution-independent parameters, so the fine λ/8 simulation mesh and the coarse λ/2 ray-cast occluder
render the *same* physical defect. Ranges are in `config.py` and tabulated in README §2.

In [ ]:
viz_setup.cross_section_overlay_figure(save_path=config.FIGURE_DIR / 'cross_sections.png');

In [ ]:
viz_setup.mesh_review_figure(save_path=config.FIGURE_DIR / 'mesh_review.png');

## 3. Fields at the metasurface plane
psi0 = direct horn term (face-independent, computed once); psi1 = single bounce off the rail (dominant);
psi2 = double bounce (≈10⁻³ of psi1). The 55° incidence produces x-fringes; short defects (crack)
visibly modulate the field along y — the signature the 3D system exploits and a 2D simulation cannot see.

In [ ]:
viz_setup.field_maps_figure(save_path=config.FIGURE_DIR / 'field_maps.png', device=DEVICE);

## 4. Trainable propagation
The exact Rayleigh–Sommerfeld kernel of Face3D's verified `prop3d`, applied via FFT linear convolution
(machine-precision identical to conv2d, ~10³× faster). The angular-spectrum propagator is kept as an
independent cross-check.

In [ ]:
viz_setup.propagator_figure(save_path=config.FIGURE_DIR / 'propagators.png');

## 5. Metasurface parameterizations — MetaUnit BLOCKED at λ=5
`SLM2D`: idealized phase-only mask (lossless, unquantized) — wavelength-agnostic, the usable
surface today. `MetaUnitSoft`: pillar-width map through the Face3D meta-atom library
(per-pixel polynomial fits, **central 60×30 crop** of the 80×80 library, sigmoid width
reparameterization keeping gradients alive at the [1, 3.8] mm bounds).

**The library is an 8 mm-band fit** and does not transfer across wavelength — and at λ=5 the
unit cell is 2.5 mm, so 3.8 mm pillars are geometrically impossible. `MetaUnitSoft` therefore
raises at λ≠8 by design; the cell below catches the refusal. The figure it would draw
documents the λ=8 library, not the current configuration.

In [ ]:
try:
    viz_setup.metaunit_figure(save_path=config.FIGURE_DIR / 'metaunit_library.png');
except RuntimeError as err:
    print('metaunit blocked (expected at lambda=%s mm):' % config.WVL)
    print(err)

## 6. Detectors
The start is a **dense 13×10 = 130-window lattice** (11.375×7.0 mm windows, 92% plane coverage) — the
tiling bound `floor(aperture/window)`, derived in config. Centers are trainable through sigmoid-edged
soft masks whose edge softness anneals during training (floored at half a pixel per axis, so position
gradients never die); evaluation always uses the hard binary windows. **Redundancy** pruning — each
detector valued by its unique contribution, `std × (1 − max|corr| to survivors)` — trims 130 → 8 during
training. Face3D's variance criterion is only valid for sparse layouts and is kept for comparison
(`prune_criterion='variance'`); V0b is the regression gate.

In [ ]:
viz_setup.detector_figure(save_path=config.FIGURE_DIR / 'detectors.png');

## 7. Barcode space (untrained)
Detector powers for one sample per class through the untrained optics, and their cosine gap vs the
intact barcode. Training uses the **rank objective** (pairwise soft-AUC on per-sample-normalized
barcodes) with the operating threshold *calibrated* on the validation intact spread, keeps the intact
cluster compact, separates class centroids, rewards light landing on the surviving windows, and trains
the tiny linear head for **crack / dent / wear / shell** classification. The legacy fixed-0.40-margin
loss survives as `objective='margin'`.

In [ ]:
viz_setup.barcode_figure(save_path=config.FIGURE_DIR / 'barcode_untrained.png', device=DEVICE);

## Approval checklist
- [ ] Setup dimensions/orientations correct (diagram §1)
- [ ] Defect depth fields per class reasonable (§2)
- [ ] Field structure plausible (§3)
- [ ] Library crop 60×30 acceptable (§5) — note `MetaUnitSoft` is BLOCKED at λ=5 until a 60 GHz
      meta-atom library is fitted, so §5 documents the λ=8 library only
- [ ] Detector layout/sizes acceptable (§6)

After approval: `python generate_dataset_3d.py --profile lab --smoke` (20/class smoke set, or
`--smoke-n 80` for geometry exploration), then the full run with
`python generate_dataset_3d.py --profile lab --name L5_H150_v1`. `rail3D_pipeline.ipynb` narrates the
whole path end to end.